# Temas Tratados en el Trabajo Práctico 4

* Representación del Conocimiento y Razonamiento Lógico.

* Estrategias de resolución de hipótesis: Encadenamiento hacia Adelante, Encadenamiento hacia Atrás y Resolución por Contradicción.

* Representación basada en circuitos.

## Ejercicios Teóricos

1. ¿Qué es una inferencia?

Inferencia: proceso de usar un modelo ya entrenado para obtener una predicción o conclusión a partir de datos nuevos que no vio durante el entrenamiento.

Entrenamiento: el modelo ajusta sus parámetros con datos conocidos.
Inferencia: el modelo entrenado recibe una entrada nueva y produce una salida (una clase, un valor, una probabilidad).
Ejemplo: una red entrenada para clasificar dígitos recibe una imagen nueva y responde "7".

En términos de lógica, inferir es derivar una conclusión a partir de premisas dadas aplicando reglas de razonamiento (deductivo, inductivo o abductivo).

2. ¿Cómo se verifica que un modelo se infiere de la base de conocimientos?

Verificar que KB ⊨ α (α se infiere de la base de conocimientos):

Significa comprobar que en todo modelo donde la base de conocimientos (KB) es verdadera, α también lo es. Métodos:

Tabla de verdad (model checking): se enumeran todas las interpretaciones posibles de los símbolos. Si en cada fila donde la KB es verdadera α también es verdadera, entonces KB ⊨ α. Correcto y completo, pero cuesta O(2ⁿ).
Demostración / prueba (theorem proving): derivar α aplicando reglas de inferencia (Modus Ponens, resolución, etc.) sin recorrer todos los modelos.
Refutación por resolución: se agrega ¬α a la KB, se pasa todo a forma clausal y se aplica resolución. Si se llega a la cláusula vacía (contradicción), entonces KB ⊨ α.

3. Observe la siguiente base de conocimiento:

$R1: b ∧ c → a$

$R2: d ∧ e → b$

$R3: g ∧ e → b$

$R4: e → c$

$R5: d$

$R6: e$

$R7: a ∧ g → f$

        3.1 ¿Cómo se puede probar que $a = True$ a través del encadenamiento hacia adelante? Este método solamente usa reglas ya incorporadas a la base de conocimiento para inferir la hipótesis, ¿qué propiedad debe tener el algoritmo para asegurar que esta inferencia sea posible?

        3.2 ¿Cómo se puede probar que $a = True$ a través del encadenamiento hacia atrás? Este método asigna un valor de verdad a la hipótesis y deriva las sentencias de la base de conocimiento, ¿qué propiedad debe tener el algoritmo para asegurar que esta derivación sea posible?

        3.3 Exprese la base de conocimiento en su Forma Normal Conjuntiva. A continuación, demuestre por contradicción que $a = True$.

### Resolución del punto 3

**Base de conocimiento**

| Regla | Sentencia |
|-------|-----------|
| R1 | $b \land c \rightarrow a$ |
| R2 | $d \land e \rightarrow b$ |
| R3 | $g \land e \rightarrow b$ |
| R4 | $e \rightarrow c$ |
| R5 | $d$ |
| R6 | $e$ |
| R7 | $a \land g \rightarrow f$ |

Hechos iniciales: $\{d,\ e\}$ (R5 y R6).

---

#### 3.1 Encadenamiento hacia adelante

Se parte de los hechos y se dispara toda regla cuyas premisas ya estén en la base, agregando el consecuente, hasta llegar a un punto fijo:

| Paso | Regla disparada | Premisas (ya conocidas) | Hecho nuevo | Base de hechos |
|------|-----------------|--------------------------|-------------|----------------|
| 0 | — | — | — | $\{d, e\}$ |
| 1 | R4: $e \rightarrow c$ | $e$ | $c$ | $\{d, e, c\}$ |
| 2 | R2: $d \land e \rightarrow b$ | $d,\ e$ | $b$ | $\{d, e, c, b\}$ |
| 3 | R1: $b \land c \rightarrow a$ | $b,\ c$ | $a$ | $\{d, e, c, b, a\}$ |

En el paso 3 se infiere $a$, por lo tanto **$a = True$**. Después no surgen hechos nuevos (R3 necesita $g$, R7 necesita $g$), y el algoritmo termina.

**Propiedad necesaria:** el encadenamiento hacia adelante es *correcto* (sound) —solo aplica Modus Ponens— y, sobre una base formada por **cláusulas definidas de Horn** (a lo sumo un literal positivo, como es este caso), es además **completo**: todo átomo que es consecuencia lógica de la KB ($KB \models a$) se agrega efectivamente a la base de hechos antes de alcanzar el punto fijo. Esa **completitud para cláusulas de Horn**, junto con la terminación garantizada (el conjunto de átomos es finito y solo crece), es lo que asegura que la inferencia de $a$ sea posible.

---

#### 3.2 Encadenamiento hacia atrás

Se toma $a$ como objetivo (se le asigna el valor a probar) y se busca hacia atrás qué reglas lo concluyen, descomponiendo cada meta en submetas (AND de antecedentes, OR entre reglas alternativas):

```
Probar a
└── R1: b ∧ c → a        (única regla con consecuente a)
    ├── Probar b
    │   └── R2: d ∧ e → b
    │       ├── Probar d  → d es HECHO (R5) ✓
    │       └── Probar e  → e es HECHO (R6) ✓
    │       ⇒ b probado ✓
    └── Probar c
        └── R4: e → c
            └── Probar e  → e es HECHO (R6) ✓
            ⇒ c probado ✓
    ⇒ a probado ✓
```

Todas las submetas se reducen a hechos, por lo tanto **$a = True$**.

**Propiedad necesaria:** al igual que el hacia adelante, el encadenamiento hacia atrás es *correcto* y **completo para cláusulas de Horn**, de modo que si $KB \models a$ existe un árbol de prueba con hojas en los hechos. Para que la derivación siempre sea posible (que el algoritmo *termine*), debe además **detectar submetas repetidas / ciclos**: llevar registro de las metas abiertas en la rama actual y podar cuando una meta se vuelve a plantear. Sin esa detección de ciclos, una KB con recursión (p. ej. $b \rightarrow b$) haría divergir la búsqueda. En esta KB el grafo de metas es acíclico, así que la derivación concluye.

---

#### 3.3 Forma Normal Conjuntiva y demostración por contradicción

**Paso a CNF** (eliminar $\rightarrow$ con $p \rightarrow q \equiv \lnot p \lor q$ y aplicar De Morgan):

| Regla | Implicación | Cláusula CNF |
|-------|-------------|--------------|
| R1 | $b \land c \rightarrow a$ | $\lnot b \lor \lnot c \lor a$ |
| R2 | $d \land e \rightarrow b$ | $\lnot d \lor \lnot e \lor b$ |
| R3 | $g \land e \rightarrow b$ | $\lnot g \lor \lnot e \lor b$ |
| R4 | $e \rightarrow c$ | $\lnot e \lor c$ |
| R5 | $d$ | $d$ |
| R6 | $e$ | $e$ |
| R7 | $a \land g \rightarrow f$ | $\lnot a \lor \lnot g \lor f$ |

KB en CNF:

$$(\lnot b \lor \lnot c \lor a) \land (\lnot d \lor \lnot e \lor b) \land (\lnot g \lor \lnot e \lor b) \land (\lnot e \lor c) \land d \land e \land (\lnot a \lor \lnot g \lor f)$$

**Refutación.** Para probar $a$ se agrega su negación $\lnot a$ al conjunto de cláusulas y se aplica resolución buscando la cláusula vacía $\square$:

| Paso | Cláusulas resueltas | Literal pivote | Resolvente |
|------|---------------------|----------------|------------|
| 1 | $d$ ,  $\ \lnot d \lor \lnot e \lor b$ | $d / \lnot d$ | $\lnot e \lor b$ |
| 2 | $e$ ,  $\ \lnot e \lor b$ | $e / \lnot e$ | $b$ |
| 3 | $e$ ,  $\ \lnot e \lor c$ | $e / \lnot e$ | $c$ |
| 4 | $b$ ,  $\ \lnot b \lor \lnot c \lor a$ | $b / \lnot b$ | $\lnot c \lor a$ |
| 5 | $c$ ,  $\ \lnot c \lor a$ | $c / \lnot c$ | $a$ |
| 6 | $a$ ,  $\ \lnot a$ | $a / \lnot a$ | $\square$ |

Se derivó la cláusula vacía, de modo que $KB \land \lnot a$ es insatisfacible y por lo tanto:

$$KB \models a \quad\Rightarrow\quad \boxed{a = True}$$


4. Diseñe con lógica proposicional basada en circuitos las proposiciones *OrientadoDerecha* y *Agente ubicado en la casilla [1,2]* para el mundo de wumpus de 4x4. Dibuje el circuito correspondiente.

# Punto 4 - Wumpus 4x4: circuito de combinaciones orientacion x posicion (t -> t+1)
# Salidas buscadas: OrientadoDerecha^{t+1} y L[1,2]^{t+1}
#
# Idea: para producir L[1,2] en t+1, para CADA casilla origen (L[1,1], L[1,3], L[2,2])
# y CADA orientacion en t se arma un AND(orientacion_t, posicion_t). Su salida entra a una
# caja de GIRO (sin giro / girar 90 / girar -90 / girar 180) que deja al agente con la
# orientacion necesaria, y de ahi a la caja de TRASLACION (mover derecha/izquierda/abajo).
# Convencion de giro anclada al ejemplo pedido:  Abajo --(girar -90)--> Derecha.
# Rama "permanecer": si ya esta en L[1,2] y OrientadoDerecha, va directo a la 3a columna.
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle

fig, ax = plt.subplots(figsize=(17, 16))
ax.set_xlim(0, 18)
ax.set_ylim(-1.8, 24.5)
ax.axis("off")

pos, patch = {}, {}

def _add(key, x, y, w, h, face, edge, text, fs, lw=1.4, rounded=True, bold=False):
    if rounded:
        p = FancyBboxPatch((x - w/2, y - h/2), w, h,
                           boxstyle="round,pad=0.06", linewidth=lw,
                           edgecolor=edge, facecolor=face)
    else:
        p = Rectangle((x - w/2, y - h/2), w, h, linewidth=lw,
                      edgecolor=edge, facecolor=face)
    ax.add_patch(p)
    ax.text(x, y, text, ha="center", va="center", fontsize=fs,
            fontweight="bold" if bold else "normal")
    pos[key] = (x, y)
    patch[key] = p

def estado(key, x, y, text, kind):
    face, edge = ("#dbe6f3", "#1f77b4") if kind == "o" else ("#dff0d8", "#2ca02c")
    _add(key, x, y, 2.7, 0.9, face, edge, text, 8)

def gate(key, x, y, symbol):
    _add(key, x, y, 0.8, 0.8, "#ececec", "#333333", symbol, 11, rounded=False)

def accion(key, x, y, text, kind):
    face, edge = ("#ffe6c7", "#e8830c") if kind == "giro" else ("#e9d9f4", "#7d3ca3")
    _add(key, x, y, 2.0, 0.82, face, edge, text, 7.5, rounded=False)

def salida(key, x, y, text):
    _add(key, x, y, 2.9, 1.0, "#fff1ac", "#222222", text, 9, lw=2.4, bold=True)

def conectar(a, b, color="#444444", lw=1.3, alpha=1.0, ls="solid"):
    ax.add_patch(FancyArrowPatch(pos[a], pos[b], patchA=patch[a], patchB=patch[b],
                                 shrinkA=1, shrinkB=1, arrowstyle="-|>", mutation_scale=11,
                                 linewidth=lw, color=color, alpha=alpha, linestyle=ls,
                                 connectionstyle="arc3,rad=0", zorder=1))

X_ST, X_AND, X_GIRO, X_MOV, X_OR, X_OUT = 1.7, 5.2, 8.4, 11.4, 13.8, 16.2

# --- Col 0: estados en t ---
estado("O_Der", X_ST, 22.4, "OrientadoDerecha", "o")
estado("O_Arr", X_ST, 20.9, "OrientadoArriba", "o")
estado("O_Izq", X_ST, 19.4, "OrientadoIzquierda", "o")
estado("O_Abj", X_ST, 17.9, "OrientadoAbajo", "o")
estado("L11", X_ST, 13.6, "L[1,1]", "p")
estado("L12", X_ST, 11.0, "L[1,2]", "p")
estado("L13", X_ST, 8.0, "L[1,3]", "p")
estado("L22", X_ST, 4.0, "L[2,2]", "p")

# --- Col 1: compuertas AND (13) ---
gate("AND_perm", X_AND, 23.2, "&")
A_y = [21.2, 19.7, 18.2, 16.7]
for k, y in zip(["A_Der", "A_Arr", "A_Izq", "A_Abj"], A_y):
    gate(k, X_AND, y, "&")
B_y = [14.2, 12.7, 11.2, 9.7]
for k, y in zip(["B_Izq", "B_Arr", "B_Abj", "B_Der"], B_y):
    gate(k, X_AND, y, "&")
C_y = [7.0, 5.5, 4.0, 2.5]
for k, y in zip(["C_Abj", "C_Der", "C_Arr", "C_Izq"], C_y):
    gate(k, X_AND, y, "&")

# --- Col 2: cajas de giro (1:1 con los AND de A/B/C) ---
giro_of = {
    "A_Der": ("gA_id", "sin giro"),   "A_Arr": ("gA_90", "girar 90"),
    "A_Izq": ("gA_180", "girar 180"), "A_Abj": ("gA_m90", "girar -90"),
    "B_Izq": ("gB_id", "sin giro"),   "B_Arr": ("gB_m90", "girar -90"),
    "B_Abj": ("gB_90", "girar 90"),   "B_Der": ("gB_180", "girar 180"),
    "C_Abj": ("gC_id", "sin giro"),   "C_Der": ("gC_90", "girar 90"),
    "C_Arr": ("gC_180", "girar 180"), "C_Izq": ("gC_m90", "girar -90"),
}
for andk, (gk, lab) in giro_of.items():
    accion(gk, X_GIRO, pos[andk][1], lab, "giro")

# --- Col 3: cajas de traslacion ---
accion("mov_der", X_MOV, sum(A_y)/4, "mover derecha", "mov")
accion("mov_izq", X_MOV, sum(B_y)/4, "mover izquierda", "mov")
accion("mov_abj", X_MOV, sum(C_y)/4, "mover abajo", "mov")

# --- Col 4: compuertas OR ---
gate("OR_orient", X_OR, 20.0, ">=1")
gate("OR_pos", X_OR, 10.5, ">=1")

# --- Col 5: salidas en t+1 ---
salida("OUT_ODer", X_OUT, 20.0, "OrientadoDerecha")
salida("OUT_L12", X_OUT, 10.5, "L[1,2]")

# ---------------- aristas ----------------
CO = {"O_Der": "#1f77b4", "O_Arr": "#17a2b8", "O_Izq": "#8c564b", "O_Abj": "#d62728"}
orient_to_and = {
    "O_Der": ["A_Der", "B_Der", "C_Der", "AND_perm"],
    "O_Arr": ["A_Arr", "B_Arr", "C_Arr"],
    "O_Izq": ["A_Izq", "B_Izq", "C_Izq"],
    "O_Abj": ["A_Abj", "B_Abj", "C_Abj"],
}
for s, ands in orient_to_and.items():
    for a in ands:
        conectar(s, a, color=CO[s], lw=1.0, alpha=0.55)

pos_to_and = {
    "L11": ["A_Der", "A_Arr", "A_Izq", "A_Abj"],
    "L13": ["B_Izq", "B_Arr", "B_Abj", "B_Der"],
    "L22": ["C_Abj", "C_Der", "C_Arr", "C_Izq"],
    "L12": ["AND_perm"],
}
for s, ands in pos_to_and.items():
    for a in ands:
        conectar(s, a, color="#2ca02c", lw=1.0, alpha=0.55)

for andk, (gk, _) in giro_of.items():          # AND -> caja de giro
    conectar(andk, gk, color="#444444", lw=1.3)

for gk, _ in giro_of.values():                  # caja de giro -> caja de mover
    dest = {"A": "mov_der", "B": "mov_izq", "C": "mov_abj"}[gk[1]]
    conectar(gk, dest, color="#7d3ca3", lw=1.2)

for gk in ["gA_id", "gA_90", "gA_180", "gA_m90"]:   # grupo A (giros) -> OR_orient
    conectar(gk, "OR_orient", color="#e8830c", lw=1.2)
conectar("AND_perm", "OR_orient", color="#666666", lw=1.1, ls="dashed", alpha=0.8)

for mk in ["mov_der", "mov_izq", "mov_abj"]:        # movs -> OR_pos
    conectar(mk, "OR_pos", color="#7d3ca3", lw=1.3)
conectar("AND_perm", "OR_pos", color="#666666", lw=1.1, ls="dashed", alpha=0.8)

conectar("OR_orient", "OUT_ODer", color="#222222", lw=1.6)
conectar("OR_pos", "OUT_L12", color="#222222", lw=1.6)

# ---------------- rotulos / leyenda ----------------
ax.text(X_ST, 23.9, "t", ha="center", fontsize=15, fontweight="bold")
ax.text(X_OUT, 23.9, "t + 1", ha="center", fontsize=15, fontweight="bold")
ax.text(X_AND, 22.0, "grupo A  .  origen L[1,1]  .  mover derecha", ha="left", fontsize=8, style="italic", color="#555")
ax.text(X_AND, 15.0, "grupo B  .  origen L[1,3]  .  mover izquierda", ha="left", fontsize=8, style="italic", color="#555")
ax.text(X_AND, 7.8, "grupo C  .  origen L[2,2]  .  mover abajo", ha="left", fontsize=8, style="italic", color="#555")

leg = [("orientacion en t", "#dbe6f3", "#1f77b4"),
       ("posicion en t", "#dff0d8", "#2ca02c"),
       ("accion: giro", "#ffe6c7", "#e8830c"),
       ("accion: traslacion", "#e9d9f4", "#7d3ca3"),
       ("compuerta AND (&) / OR (>=1)", "#ececec", "#333333")]
for i, (txt, fc, ec) in enumerate(leg):
    ax.add_patch(Rectangle((0.5 + i*3.5, 0.3), 0.5, 0.5, facecolor=fc, edgecolor=ec, linewidth=1.3))
    ax.text(1.1 + i*3.5, 0.55, txt, ha="left", va="center", fontsize=7.3)

ax.set_title("Punto 4 - Wumpus 4x4 : combinaciones orientacion x posicion  (t -> t+1)\n"
             "salidas: OrientadoDerecha  y  L[1,2]", fontsize=12)
ax.text(9, -1.2,
        "Nota: el update de orientacion es en rigor independiente de la posicion; "
        "el diagrama muestra las combinaciones relevantes para producir L[1,2] en t+1.",
        ha="center", va="center", fontsize=7, style="italic", color="#666")

plt.tight_layout()
plt.show()


5. El nonograma es un juego en el cual se posee un tablero en blanco y cada fila y columna presenta información sobre la longitud de un bloque en dicha fila/columna. Además, la leyenda puede indicar más de un número, indicando esto que existen varios bloques de las longitudes mostradas por la leyenda y en el mismo orden, separados por al menos un espacio vacío.
Resuelva el nonograma de la imagen de abajo escribiendo en primer lugar cada regla que puede incorporarse a la base de conocimientos inicial e incorporando cada inferencia que realice

NONOGRAMA 4x4 - Punto 5

Notacion: Crc = casilla de la fila r y columna c.  ∧ = AND , ∨ = OR.
Verdadero = casilla llena.  Falso = casilla vacia.
Leyendas filas:    R1="1 1"  R2="4"  R3="2 1"  R4="3"
Leyendas columnas: C1="3"    C2="3"  C3="2 1"  C4="3"

=== REGLAS DE FILAS (base de conocimiento inicial) ===
R1 (fila 1, "1 1"):  (C11 ∧ C13)  ∨  (C11 ∧ C14)  ∨  (C12 ∧ C14)
R2 (fila 2, "4"):    C21 ∧ C22 ∧ C23 ∧ C24
R3 (fila 3, "2 1"):  C31 ∧ C32 ∧ (NO C33) ∧ C34        [unica ubicacion en 4 celdas]
R4 (fila 4, "3"):    (C41 ∧ C42 ∧ C43)  ∨  (C42 ∧ C43 ∧ C44)

=== REGLAS DE COLUMNAS ===
C1 (col 1, "3"):    (C11 ∧ C21 ∧ C31)  ∨  (C21 ∧ C31 ∧ C41)
C2 (col 2, "3"):    (C12 ∧ C22 ∧ C32)  ∨  (C22 ∧ C32 ∧ C42)
C3 (col 3, "2 1"):  C13 ∧ C23 ∧ (NO C33) ∧ C43         [unica ubicacion en 4 celdas]
C4 (col 4, "3"):    (C14 ∧ C24 ∧ C34)  ∨  (C24 ∧ C34 ∧ C44)

=== INFERENCIAS ===
I1  (de R2): C21, C22, C23, C24 = verdaderos  (fila 2 completa).

I2  (de R3): "2 1" en 4 celdas tiene una sola ubicacion posible
     (bloque de 2 en col 1-2, hueco en col 3, bloque de 1 en col 4):
     C31, C32, C34 = verdaderos ;  C33 = falso.

I3  (de C3): "2 1" en 4 celdas, una sola ubicacion posible
     (bloque de 2 en fila 1-2, hueco en fila 3, bloque de 1 en fila 4):
     C13, C23, C43 = verdaderos ;  C33 = falso  (coincide con I2).

I4  (de R1 + I3): como C13 es verdadero, la unica opcion de R1 compatible es (C11 ∧ C13):
     C11 = verdadero ;  C12 = falso ;  C14 = falso.

I5  (de C1 + I4): como C11 es verdadero, la unica opcion de C1 compatible es (C11 ∧ C21 ∧ C31):
     C31 = verdadero (coincide con I2) ;  C41 = falso.

I6  (de R4 + I5): como C41 es falso, la unica opcion de R4 compatible es (C42 ∧ C43 ∧ C44):
     C42, C43, C44 = verdaderos.

I7  (de C2 + I4): como C12 es falso, la unica opcion de C2 compatible es (C22 ∧ C32 ∧ C42):
     C22, C32, C42 = verdaderos  (consistente con lo ya inferido).

I8  (de C4 + I4): como C14 es falso, la unica opcion de C4 compatible es (C24 ∧ C34 ∧ C44):
     C24, C34, C44 = verdaderos  (consistente con lo ya inferido).

=== SOLUCION FINAL ===
Llenas (verdadero): C11, C13, C21, C22, C23, C24, C31, C32, C34, C42, C43, C44
Vacias (falso):     C12, C14, C33, C41

        c1 c2 c3 c4     leyenda fila
 f1  [  X  .  X  .  ]     1 1
 f2  [  X  X  X  X  ]     4
 f3  [  X  X  .  X  ]     2 1
 f4  [  .  X  X  X  ]     3
 col    3  3  2.1 3


## Ejercicios de Implementación

> Recuerde adjuntar en la presentación el prompt inicial que ha utilizado para cada ejercicio de implementación y si considera que le dio la información completa para resolver el ejercicio o qué cambios adicionales tuvo que pedir de manera iterativa.

6. Implementar un motor de inferencia con encadenamiento hacia adelante. Pruébelo con las proposiciones del ejercicio 3.

In [6]:
from dataclasses import dataclass
from typing import Set, List


@dataclass(frozen=True)
class Rule:
    """Representa una regla de producción (cláusula de Horn): antecedentes -> consecuente."""
    antecedents: frozenset
    consequent: str

    def __str__(self) -> str:
        premisas = " ∧ ".join(sorted(self.antecedents)) if self.antecedents else "True"
        return f"{premisas} → {self.consequent}"


class KnowledgeBase:
    """Base de Reglas / Conocimiento: almacena las reglas de producción del sistema."""

    def __init__(self) -> None:
        self.rules: List[Rule] = []

    def add_rule(self, antecedents: Set[str], consequent: str) -> None:
        """Agrega una regla antecedentes -> consecuente (antecedentes vacío = hecho)."""
        self.rules.append(Rule(frozenset(antecedents), consequent))

    def add_fact(self, fact: str) -> None:
        """Agrega un hecho inicial, modelado como una regla sin antecedentes."""
        self.add_rule(set(), fact)


class WorkingMemory:
    """Base de Hechos: almacena las proposiciones ya demostradas como verdaderas."""

    def __init__(self) -> None:
        self.facts: Set[str] = set()

    def is_known(self, fact: str) -> bool:
        return fact in self.facts

    def add(self, fact: str) -> None:
        self.facts.add(fact)


class ForwardChainingEngine:
    """Motor de inferencia por Encadenamiento hacia Adelante (Modus Ponens iterativo).

    Dispara toda regla cuyas premisas ya estén satisfechas, incorporando el
    nuevo hecho a la memoria de trabajo, hasta alcanzar un punto fijo (no
    surgen más hechos nuevos).
    """

    def __init__(self, kb: KnowledgeBase) -> None:
        self.kb = kb
        self.wm = WorkingMemory()

    def run(self, verbose: bool = True) -> WorkingMemory:
        """Ejecuta el encadenamiento hacia adelante hasta el punto fijo."""
        if verbose:
            print("=== Traza de Encadenamiento hacia Adelante ===")

        changed = True
        ciclo = 0
        while changed:
            changed = False
            ciclo += 1
            for rule in self.kb.rules:
                if rule.consequent in self.wm.facts:
                    continue
                if rule.antecedents.issubset(self.wm.facts):
                    self.wm.add(rule.consequent)
                    changed = True
                    if verbose:
                        print(f"Ciclo {ciclo}: se dispara [{rule}]  =>  nuevo hecho '{rule.consequent}'")

        if verbose:
            print(f"Punto fijo alcanzado. Hechos deducidos: {sorted(self.wm.facts)}")
        return self.wm

    def is_true(self, query: str) -> bool:
        """Consulta si `query` puede deducirse de la base de conocimiento. Imprime el veredicto."""
        resultado = query in self.wm.facts
        veredicto = "VERDADERA" if resultado else "NO se puede inferir"
        print(f"\n¿Es '{query}' verdadera? -> {veredicto}")
        return resultado


# ---------------------------------------------------------------------------
# Caso de prueba: base de conocimiento del ejercicio 3
#   R1: b ∧ c -> a
#   R2: d ∧ e -> b
#   R3: g ∧ e -> b
#   R4: e -> c
#   R5: d           (hecho)
#   R6: e           (hecho)
#   R7: a ∧ g -> f
# ---------------------------------------------------------------------------

kb = KnowledgeBase()
kb.add_rule({"b", "c"}, "a")   # R1
kb.add_rule({"d", "e"}, "b")   # R2
kb.add_rule({"g", "e"}, "b")   # R3
kb.add_rule({"e"}, "c")        # R4
kb.add_fact("d")               # R5
kb.add_fact("e")               # R6
kb.add_rule({"a", "g"}, "f")   # R7

engine = ForwardChainingEngine(kb)
engine.run()

# ---------------------------------------------------------------------------
# Consulta interactiva: el usuario elige qué proposición quiere verificar
# ---------------------------------------------------------------------------
proposiciones_validas = {"a", "b", "c", "d", "e", "f", "g"}

while True:
    query = input("¿Qué proposición querés consultar (a/b/c/d/e/f/g)? ").strip().lower()
    if query in proposiciones_validas:
        break
    print(f"'{query}' no es una proposición válida. Elegí entre: {sorted(proposiciones_validas)}")

engine.is_true(query)

# Si se consulta 'f', se explica por qué no puede inferirse sin el hecho 'g'
if query == "f" and not engine.wm.is_known("f"):
    print("\nAnálisis de 'f': la regla R7 (a ∧ g -> f) requiere 'g', que nunca")
    print("fue incorporado como hecho ni pudo deducirse de otra regla, por lo")
    print("que 'f' permanece indemostrable con la base de conocimiento actual.")


=== Traza de Encadenamiento hacia Adelante ===
Ciclo 1: se dispara [True → d]  =>  nuevo hecho 'd'
Ciclo 1: se dispara [True → e]  =>  nuevo hecho 'e'
Ciclo 2: se dispara [d ∧ e → b]  =>  nuevo hecho 'b'
Ciclo 2: se dispara [e → c]  =>  nuevo hecho 'c'
Ciclo 3: se dispara [b ∧ c → a]  =>  nuevo hecho 'a'
Punto fijo alcanzado. Hechos deducidos: ['a', 'b', 'c', 'd', 'e']

¿Es 'd' verdadera? -> VERDADERA


7. Implementar un motor de inferencia con encadenamiento hacia atrás. Pruébelo con las proposiciones del ejercicio 3.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, FrozenSet, List, Optional, Set


# ===========================================================================
# 1. BASE DE CONOCIMIENTO  (separación estructural: Hechos vs. Reglas)
# ===========================================================================

@dataclass(frozen=True)
class Rule:
    """Cláusula de Horn: `antecedents` (submetas en AND) → `consequent` (meta)."""
    antecedents: FrozenSet[str]
    consequent: str
    name: str = ""

    def __str__(self) -> str:
        premisas = " ∧ ".join(sorted(self.antecedents)) if self.antecedents else "True"
        etiqueta = f"{self.name}: " if self.name else ""
        return f"{etiqueta}{premisas} → {self.consequent}"


class FactBase:
    """Base de Hechos: conjunto de proposiciones confirmadas como verdaderas."""

    def __init__(self, facts: Optional[Set[str]] = None) -> None:
        self._facts: Set[str] = set(facts or set())

    def is_fact(self, prop: str) -> bool:
        return prop in self._facts

    def add(self, prop: str) -> None:
        self._facts.add(prop)

    def __repr__(self) -> str:
        return f"FactBase({sorted(self._facts)})"


class RuleBase:
    """Base de Reglas: indexa cada regla por su consecuente (meta) para
    localizar en O(1) todas las reglas capaces de concluir un objetivo dado.
    Varias reglas para una misma meta forman un nodo OR."""

    def __init__(self) -> None:
        self._by_consequent: Dict[str, List[Rule]] = {}

    def add_rule(self, antecedents: Set[str], consequent: str, name: str = "") -> None:
        rule = Rule(frozenset(antecedents), consequent, name)
        self._by_consequent.setdefault(consequent, []).append(rule)

    def rules_for(self, goal: str) -> List[Rule]:
        """Reglas cuyo consecuente es `goal` (alternativas OR para probarlo)."""
        return self._by_consequent.get(goal, [])


# ===========================================================================
# 2. MOTOR DE INFERENCIA  ·  Encadenamiento hacia Atrás (Goal-Driven)
# ===========================================================================

class BackwardChainingEngine:
    """Razonamiento dirigido por objetivos.

    Para probar una meta:
      1. Si es un hecho conocido  -> True.
      2. Si ya está en la rama actual (`visited`) -> ciclo, se poda -> False.
      3. Se buscan las reglas cuyo consecuente sea la meta (OR); una regla
         tiene éxito si TODOS sus antecedentes se prueban recursivamente (AND).
    Cada paso se imprime indentado, formando el árbol de deducción.
    """

    def __init__(self, facts: FactBase, rules: RuleBase) -> None:
        self.facts = facts
        self.rules = rules

    def prove(self, goal: str, visited: Optional[FrozenSet[str]] = None,
              depth: int = 0) -> bool:
        if visited is None:
            visited = frozenset()
        pad = "    " * depth

        # --- 1. ¿Hecho conocido? -------------------------------------------
        if self.facts.is_fact(goal):
            print(f"{pad}✓ '{goal}' es un HECHO")
            return True

        # --- 2. Control de ciclos en la rama actual -----------------------
        if goal in visited:
            print(f"{pad}✗ '{goal}' ya está en la rama actual → CICLO, se poda")
            return False

        # --- 3. Reglas que concluyen la meta (nodo OR) -------------------
        candidatas = self.rules.rules_for(goal)
        if not candidatas:
            print(f"{pad}✗ '{goal}' no es hecho y ninguna regla lo concluye "
                  f"→ submeta sin respaldo")
            return False

        visited = visited | {goal}
        print(f"{pad}? meta '{goal}': {len(candidatas)} regla(s) candidata(s)")

        for rule in candidatas:
            print(f"{pad}  ├─ intento [{rule}]")
            # nodo AND: se corta ante el primer antecedente que falla
            exito = True
            for ante in sorted(rule.antecedents):
                if not self.prove(ante, visited, depth + 2):
                    print(f"{pad}  │  └─ falla la submeta '{ante}' → se descarta [{rule.name or rule}]")
                    exito = False
                    break
            if exito:
                print(f"{pad}  └─ ✓ '{goal}' PROBADA por [{rule}]")
                return True

        print(f"{pad}  └─ ✗ '{goal}' NO se pudo probar con ninguna regla")
        return False

    def query(self, goal: str) -> bool:
        print(f"\n{'=' * 68}\nCONSULTA:  ¿ '{goal}' = True ?\n{'=' * 68}")
        resultado = self.prove(goal)
        print(f"{'-' * 68}\nRESULTADO:  {goal} = {resultado}\n{'-' * 68}")
        return resultado


# ===========================================================================
# 3. CASO DE PRUEBA OBLIGATORIO  (base de conocimiento del ejercicio 3)
#     R1: b ∧ c → a      R2: d ∧ e → b      R3: g ∧ e → b
#     R4: e → c          R7: a ∧ g → f      R5: d  ·  R6: e   (hechos)
# ===========================================================================

hechos = FactBase()
hechos.add("d")                              # R5
hechos.add("e")                              # R6

reglas = RuleBase()
reglas.add_rule({"b", "c"}, "a", "R1")
reglas.add_rule({"d", "e"}, "b", "R2")
reglas.add_rule({"g", "e"}, "b", "R3")
reglas.add_rule({"e"}, "c", "R4")
reglas.add_rule({"a", "g"}, "f", "R7")

motor = BackwardChainingEngine(hechos, reglas)

# --- Consulta por 'a'  → se espera True -----------------------------------
#     a → [b, c] → b por R2 (d ∧ e, ambos hechos) ; c por R4 (e, hecho)
motor.query("a")

# --- Consulta por 'f'  → se espera False --------------------------------
#     f por R7 exige (a ∧ g). 'a' se prueba, pero 'g' no es hecho ni hay
#     regla que lo concluya, así que R7 no se dispara y 'f' queda sin probar.
motor.query("f")
print("\nExplicación de 'f': la única regla para 'f' es R7 (a ∧ g → f).")
print("La submeta 'a' se demuestra, pero 'g' no figura como hecho y ninguna")
print("regla lo tiene como consecuente: es una submeta sin respaldo, por lo")
print("que R7 nunca se satisface y 'f' es indemostrable con esta base.")


# ===========================================================================
# 4. CONSULTA INTERACTIVA
# ===========================================================================
proposiciones_validas = {"a", "b", "c", "d", "e", "f", "g"}

try:
    while True:
        consulta = input("\n¿Qué proposición querés consultar (a/b/c/d/e/f/g)? ").strip().lower()
        if consulta in proposiciones_validas:
            motor.query(consulta)
            break
        print(f"'{consulta}' no es válida. Elegí entre: {sorted(proposiciones_validas)}")
except EOFError:
    # Ejecución no interactiva (p. ej. 'Run All'): se omite la consulta manual.
    print("\n[Entrada interactiva no disponible: se muestran solo las consultas por 'a' y 'f'.]")


8. Implementar un motor de inferencia por contradicción que detecte si el conjunto de proposiciones del ejercicio 3 es inconsistente.

In [ ]:
"""
Ejercicio 8 - Motor de Inferencia por Contradiccion
===================================================
Algoritmo de RESOLUCION PROPOSICIONAL con REFUTACION (Resolution by Refutation).

El motor cumple dos funciones:

  A) Detectar si la Base de Conocimiento (KB) es INCONSISTENTE por si misma,
     es decir, si a partir de sus clausulas se puede derivar la clausula
     vacia  []  (contradiccion).

  B) Responder consultas del usuario sobre una hipotesis alpha usando
     REFUTACION:  se agrega  ¬alpha  a la KB y se verifica si
          KB  ∧  ¬alpha   ⊢   []
     Si se deriva la clausula vacia, entonces  KB ⊨ alpha  (alpha es verdadera).

Arquitectura modular (todo dentro de la celda, separado por secciones):

  1. AST de formulas .............. Var / Not / And / Or / Implies / Iff
  2. Conversion a Forma Normal Conjuntiva (CNF)
       2.1 eliminacion de bicondicional : p ↔ q  ≡  (p → q) ∧ (q → p)
       2.2 eliminacion de implicacion   : p → q  ≡  ¬p ∨ q
       2.3 De Morgan / doble negacion   : ¬(p ∧ q) ≡ ¬p ∨ ¬q ; ¬(p ∨ q) ≡ ¬p ∧ ¬q ; ¬¬p ≡ p
       2.4 distribucion de ∨ sobre ∧    : p ∨ (q ∧ r) ≡ (p ∨ q) ∧ (p ∨ r)
  3. Clausulas como conjuntos (frozenset) de literales
  4. Resolucion binaria + refutacion, con traza paso a paso
  5. Casos de validacion obligatorios (base del ejercicio 3)
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, FrozenSet, List, Optional, Set, Tuple


# ===========================================================================
# 1. AST DE FORMULAS PROPOSICIONALES
# ===========================================================================
# Cada conectiva es un nodo inmutable. Al ser dataclasses "frozen", dos nodos
# con la misma estructura son iguales y hasheables: eso permite comparar
# formulas por valor (util para iterar la CNF hasta punto fijo).

class Formula:
    """Clase base marcadora para todos los nodos del arbol de formula."""


@dataclass(frozen=True)
class Var(Formula):
    """Variable / atomo proposicional (p. ej. 'a')."""
    name: str


@dataclass(frozen=True)
class Not(Formula):
    """Negacion  ¬f."""
    f: Formula


@dataclass(frozen=True)
class And(Formula):
    """Conjuncion  left ∧ right."""
    left: Formula
    right: Formula


@dataclass(frozen=True)
class Or(Formula):
    """Disyuncion  left ∨ right."""
    left: Formula
    right: Formula


@dataclass(frozen=True)
class Implies(Formula):
    """Implicacion  left → right."""
    left: Formula
    right: Formula


@dataclass(frozen=True)
class Iff(Formula):
    """Bicondicional  left ↔ right."""
    left: Formula
    right: Formula


# ===========================================================================
# 2. CONVERSION A FORMA NORMAL CONJUNTIVA (CNF)
# ===========================================================================
# La CNF es una conjuncion (AND) de clausulas, donde cada clausula es una
# disyuncion (OR) de literales. Se obtiene aplicando en orden las
# equivalencias logicas 2.1 -> 2.4.

def _elim_iff(f: Formula) -> Formula:
    """2.1  Reemplaza  p ↔ q  por  (p → q) ∧ (q → p)."""
    if isinstance(f, Iff):
        l, r = _elim_iff(f.left), _elim_iff(f.right)
        return And(Implies(l, r), Implies(r, l))
    if isinstance(f, Implies):
        return Implies(_elim_iff(f.left), _elim_iff(f.right))
    if isinstance(f, And):
        return And(_elim_iff(f.left), _elim_iff(f.right))
    if isinstance(f, Or):
        return Or(_elim_iff(f.left), _elim_iff(f.right))
    if isinstance(f, Not):
        return Not(_elim_iff(f.f))
    return f  # Var


def _elim_implies(f: Formula) -> Formula:
    """2.2  Reemplaza  p → q  por  ¬p ∨ q."""
    if isinstance(f, Implies):
        return Or(Not(_elim_implies(f.left)), _elim_implies(f.right))
    if isinstance(f, And):
        return And(_elim_implies(f.left), _elim_implies(f.right))
    if isinstance(f, Or):
        return Or(_elim_implies(f.left), _elim_implies(f.right))
    if isinstance(f, Not):
        return Not(_elim_implies(f.f))
    return f  # Var


def _push_not(f: Formula) -> Formula:
    """2.3  Empuja las negaciones hasta los atomos (De Morgan + doble negacion).

    Precondicion: la formula ya no contiene Implies ni Iff.
    """
    if isinstance(f, Not):
        g = f.f
        if isinstance(g, Var):
            return f                                   # ¬atomo : ya es literal
        if isinstance(g, Not):
            return _push_not(g.f)                      # ¬¬p ≡ p
        if isinstance(g, And):
            return Or(_push_not(Not(g.left)), _push_not(Not(g.right)))   # ¬(p∧q) ≡ ¬p∨¬q
        if isinstance(g, Or):
            return And(_push_not(Not(g.left)), _push_not(Not(g.right)))  # ¬(p∨q) ≡ ¬p∧¬q
    if isinstance(f, And):
        return And(_push_not(f.left), _push_not(f.right))
    if isinstance(f, Or):
        return Or(_push_not(f.left), _push_not(f.right))
    return f  # Var


def _distribuir(f: Formula) -> Formula:
    """2.4  Distribuye ∨ sobre ∧ :  p ∨ (q ∧ r) ≡ (p ∨ q) ∧ (p ∨ r).

    Se aplica una pasada; el llamador la repite hasta punto fijo.
    """
    if isinstance(f, And):
        return And(_distribuir(f.left), _distribuir(f.right))
    if isinstance(f, Or):
        l, r = _distribuir(f.left), _distribuir(f.right)
        if isinstance(l, And):
            return And(_distribuir(Or(l.left, r)), _distribuir(Or(l.right, r)))
        if isinstance(r, And):
            return And(_distribuir(Or(l, r.left)), _distribuir(Or(l, r.right)))
        return Or(l, r)
    return f  # Var / Not(Var)


def a_cnf(f: Formula) -> Formula:
    """Devuelve la formula en CNF aplicando 2.1 -> 2.4 (distribucion iterada)."""
    f = _push_not(_elim_implies(_elim_iff(f)))
    anterior = None
    while anterior != f:               # punto fijo: la CNF ya no cambia
        anterior = f
        f = _distribuir(f)
    return f


# ===========================================================================
# 3. CLAUSULAS COMO CONJUNTOS DE LITERALES
# ===========================================================================

@dataclass(frozen=True)
class Literal:
    """Literal proposicional: un atomo, positivo (neg=False) o negado (neg=True)."""
    name: str
    neg: bool = False

    def __neg__(self) -> "Literal":
        """Literal complementario:  -a == ¬a  ;  -(¬a) == a."""
        return Literal(self.name, not self.neg)

    def __str__(self) -> str:
        return ("¬" if self.neg else "") + self.name


# Una clausula es un frozenset[Literal] (disyuncion de sus literales).
Clausula = FrozenSet[Literal]


def es_tautologia(literales: Set[Literal]) -> bool:
    """True si la clausula contiene un literal y su complemento (l ∨ ¬l ≡ True)."""
    return any(Literal(l.name, not l.neg) in literales for l in literales)


def _aplanar(f: Formula, conectiva) -> List[Formula]:
    """Aplana un arbol binario de And/Or en la lista de sus operandos."""
    if isinstance(f, conectiva):
        return _aplanar(f.left, conectiva) + _aplanar(f.right, conectiva)
    return [f]


def formula_a_clausulas(f: Formula) -> Set[Clausula]:
    """Convierte una formula a CNF y extrae su conjunto de clausulas.

    Las tautologias se descartan (no aportan informacion a la refutacion).
    """
    cnf = a_cnf(f)
    clausulas: Set[Clausula] = set()
    for conjunto in _aplanar(cnf, And):            # cada conjunto = una clausula
        literales: Set[Literal] = set()
        for disy in _aplanar(conjunto, Or):        # cada disy = un literal
            if isinstance(disy, Var):
                literales.add(Literal(disy.name, False))
            elif isinstance(disy, Not) and isinstance(disy.f, Var):
                literales.add(Literal(disy.f.name, True))
            else:
                raise ValueError(f"Elemento no literal tras CNF: {disy!r}")
        if not es_tautologia(literales):
            clausulas.add(frozenset(literales))
    return clausulas


def fmt_clausula(cl: Clausula) -> str:
    """Representacion legible de una clausula ('[]' para la clausula vacia)."""
    if not cl:
        return "[]  (clausula vacia)"
    orden = sorted(cl, key=lambda l: (l.name, l.neg))
    return "{" + " ∨ ".join(str(l) for l in orden) + "}"


# ===========================================================================
# 4. BASE DE CONOCIMIENTO
# ===========================================================================

class BaseConocimiento:
    """Almacena las formulas originales y su traduccion a clausulas CNF.

    Se guarda el 'origen' (R1, R2, ...) de cada clausula para que la traza
    sea comprensible.
    """

    def __init__(self) -> None:
        self.formulas: List[Tuple[str, Formula]] = []
        self.clausulas: List[Tuple[str, Clausula]] = []

    def agregar(self, nombre: str, formula: Formula) -> None:
        """Incorpora una regla o hecho a la base y lo pasa a forma clausal."""
        self.formulas.append((nombre, formula))
        for cl in formula_a_clausulas(formula):
            self.clausulas.append((nombre, cl))

    def conjunto_clausulas(self) -> Set[Clausula]:
        """Conjunto (sin repetidos) de todas las clausulas de la base."""
        return {cl for _, cl in self.clausulas}

    def copia(self) -> "BaseConocimiento":
        """Copia independiente (para poder 'ensuciar' la base sin efectos)."""
        nueva = BaseConocimiento()
        for nombre, formula in self.formulas:
            nueva.agregar(nombre, formula)
        return nueva

    def imprimir(self) -> None:
        print("Clausulas CNF de la base de conocimiento:")
        for origen, cl in self.clausulas:
            print(f"   {origen:>6} :  {fmt_clausula(cl)}")


# ===========================================================================
# 5. MOTOR DE RESOLUCION BINARIA + REFUTACION
# ===========================================================================

# Cada clausula derivada apunta a sus 'padres': (c1, c2, literal_pivote).
# Las clausulas axioma (de la KB) tienen padre None.
Padres = Dict[Clausula, Optional[Tuple[Clausula, Clausula, Literal]]]


def resolventes(c1: Clausula, c2: Clausula):
    """Genera todas las resolventes binarias de c1 y c2.

    Para cada literal  l ∈ c1  cuyo complemento  ¬l ∈ c2, la resolvente es
        (c1 \\ {l})  ∪  (c2 \\ {¬l})
    Se descartan las resolventes tautologicas (no aportan a la refutacion).
    Devuelve pares (resolvente, literal_pivote).
    """
    for l in c1:
        if -l in c2:
            nueva = (c1 - {l}) | (c2 - {-l})
            if es_tautologia(nueva):
                continue
            yield frozenset(nueva), l


class MotorResolucion:
    """Motor de inferencia por contradiccion (resolucion + refutacion)."""

    def __init__(self, kb: BaseConocimiento) -> None:
        self.kb = kb

    # ---- nucleo: intenta derivar la clausula vacia --------------------------
    def _refutar(self, extra: Set[Clausula]) -> Tuple[bool, List[Tuple]]:
        """Satura por resolucion  KB ∪ extra.

        Return: (se_derivo_vacia, pasos_de_la_traza).
        'pasos' es la secuencia minima (c1, c2, pivote, resolvente) que
        conduce a la clausula vacia.
        """
        clausulas: Set[Clausula] = self.kb.conjunto_clausulas() | set(extra)
        padres: Padres = {cl: None for cl in clausulas}

        # La KB podria contener ya la clausula vacia (caso degenerado).
        if frozenset() in clausulas:
            return True, []

        while True:
            nuevas: Set[Clausula] = set()
            lista = list(clausulas)
            for i in range(len(lista)):
                for j in range(i + 1, len(lista)):
                    ci, cj = lista[i], lista[j]
                    for resolvente, pivote in resolventes(ci, cj):
                        if resolvente in clausulas or resolvente in nuevas:
                            continue
                        padres[resolvente] = (ci, cj, pivote)
                        if not resolvente:                       # clausula vacia -> contradiccion
                            return True, self._traza(resolvente, padres)
                        nuevas.add(resolvente)

            if not nuevas:                                       # saturacion: nada nuevo
                return False, []
            clausulas |= nuevas

    @staticmethod
    def _traza(objetivo: Clausula, padres: Padres) -> List[Tuple]:
        """Reconstruye, en orden de aplicacion, los pasos que llevan a `objetivo`."""
        pasos: List[Tuple] = []
        vistos: Set[Clausula] = set()

        def visitar(cl: Clausula) -> None:
            if cl in vistos:
                return
            vistos.add(cl)
            info = padres.get(cl)
            if info is None:            # es un axioma de la KB: no genera paso
                return
            c1, c2, pivote = info
            visitar(c1)
            visitar(c2)
            pasos.append((c1, c2, pivote, cl))

        visitar(objetivo)
        return pasos

    @staticmethod
    def _imprimir_traza(pasos: List[Tuple]) -> None:
        if not pasos:
            print("   (la clausula vacia ya estaba presente en la base)")
            return
        print(f"   Traza de resolucion ({len(pasos)} paso(s) hasta  []):")
        for k, (c1, c2, pivote, res) in enumerate(pasos, 1):
            print(f"    {k:>2}.  {fmt_clausula(c1)}   +   {fmt_clausula(c2)}")
            print(f"         resolviendo sobre  {pivote} / {-pivote}   =>   {fmt_clausula(res)}")

    # ---- A) consistencia de la base ---------------------------------------
    def es_consistente(self) -> bool:
        """Reporta si la KB es consistente. Return True si lo es."""
        contradiccion, pasos = self._refutar(extra=set())
        if contradiccion:
            print(">>> BASE INCONSISTENTE: se derivo la clausula vacia  []")
            self._imprimir_traza(pasos)
            return False
        print(">>> Base consistente, no hay contradiccion")
        return True

    # ---- B) consulta de una hipotesis alpha por refutacion ----------------
    def consultar(self, alpha: Formula, etiqueta: str = "alpha") -> bool:
        """Verifica  KB ⊨ alpha  agregando  ¬alpha  y buscando  [].

        Return True si alpha queda demostrada.
        """
        neg_alpha = formula_a_clausulas(Not(alpha))
        print(f"Consulta:  ¿ {etiqueta} = True ?   (se agrega ¬{etiqueta} a la base)")
        print("   Clausulas de  ¬" + etiqueta + ":  "
              + ", ".join(fmt_clausula(c) for c in neg_alpha))

        # Antes de refutar, conviene saber si la propia KB ya es inconsistente:
        # en ese caso 'probaria' cualquier cosa de forma espuria.
        if not self._refutar(extra=set())[0]:
            contradiccion, pasos = self._refutar(extra=neg_alpha)
            if contradiccion:
                print(f">>> {etiqueta} = True   (KB ∧ ¬{etiqueta} ⊢ [], refutacion exitosa)")
                self._imprimir_traza(pasos)
                return True
            print(f">>> {etiqueta} NO se infiere de la base "
                  f"(no se pudo derivar  []  a partir de KB ∧ ¬{etiqueta})")
            return False

        print(">>> La KB ya es inconsistente: la consulta no es informativa "
              "hasta resolver la contradiccion interna.")
        return False


# ===========================================================================
# 6. CASOS DE VALIDACION OBLIGATORIOS  (base del ejercicio 3)
# ===========================================================================
#   R1: b ∧ c → a        R2: d ∧ e → b        R3: g ∧ e → b
#   R4: e → c            R5: d  (hecho)        R6: e  (hecho)
#   R7: a ∧ g → f
# ---------------------------------------------------------------------------

a, b, c, d, e, g, h_f = Var("a"), Var("b"), Var("c"), Var("d"), Var("e"), Var("g"), Var("f")

kb = BaseConocimiento()
kb.agregar("R1", Implies(And(b, c), a))
kb.agregar("R2", Implies(And(d, e), b))
kb.agregar("R3", Implies(And(g, e), b))
kb.agregar("R4", Implies(e, c))
kb.agregar("R5", d)                       # hecho
kb.agregar("R6", e)                       # hecho
kb.agregar("R7", Implies(And(a, g), h_f))

print("=" * 72)
kb.imprimir()

motor = MotorResolucion(kb)

# ---- CASO 1: consistencia de la base original --------------------------------
print("\n" + "=" * 72)
print("[CASO 1]  Consistencia de la base original")
print("-" * 72)
motor.es_consistente()

# ---- CASO 2: consultar si 'a' es verdadero ---------------------------------
print("\n" + "=" * 72)
print("[CASO 2]  Consulta por refutacion:  ¿ a = True ?")
print("-" * 72)
motor.consultar(a, "a")

# ---- CASO 3: base inconsistente forzada (se inserta el hecho ¬d) -----------
print("\n" + "=" * 72)
print("[CASO 3]  Se fuerza un hecho contradictorio:  ¬d  (la base ya afirma d en R5)")
print("-" * 72)
kb_inconsistente = kb.copia()
kb_inconsistente.agregar("EXTRA", Not(d))
MotorResolucion(kb_inconsistente).es_consistente()

# ---- Extra didactico: 'f' NO deberia inferirse (falta el hecho 'g') --------
print("\n" + "=" * 72)
print("[EXTRA]  Consulta por refutacion:  ¿ f = True ?   (se espera que NO)")
print("-" * 72)
motor.consultar(h_f, "f")
print("\nMotivo: la unica regla para 'f' es R7 (a ∧ g → f). 'a' se demuestra,")
print("pero 'g' no es hecho ni consecuente de ninguna regla, por lo que")
print("KB ∧ ¬f resulta SATISFACIBLE (satura sin producir la clausula vacia).")


# Bibliografía

[Russell, S. & Norvig, P. (2004) _Inteligencia Artificial: Un Enfoque Moderno_. Pearson Educación S.A. (2a Ed.) Madrid, España](https://www.academia.edu/8241613/Inteligencia_Aritificial_Un_Enfoque_Moderno_2da_Edici%C3%B3n_Stuart_J_Russell_y_Peter_Norvig)

[Poole, D. & Mackworth, A. (2023) _Artificial Intelligence: Foundations of Computational Agents_. Cambridge University Press (3a Ed.) Vancouver, Canada](https://artint.info/3e/html/ArtInt3e.html)